# RAG Evaluation & Experimentation — Phase 2

Phase 1 built a *working* RAG pipeline on SEC filings. This notebook makes it a **measured**
one — the evaluation / guardrails / monitoring feedback loop that most RAG tutorials skip, and
the part that proves what you ship actually works.

```
        Phase 1 pipeline                          Phase 2 (this notebook)
01→02→03→04→05  ──────────────▶   gold Q/A set → retrieval metrics → LLM-judge → 1 experiment
                                   (recall@k / MRR)  (faithfulness)   (change one knob, measure)
```

### The stack (hybrid)
- **Generation:** local **Mistral-7B** (4-bit) — the pipeline we're evaluating.
- **Judge:** the **Claude API** (`claude-opus-4-8`) — a stronger model grades faithfulness &
  relevance. A 7B model grading its own output is a weak judge; this is more credible.

### Memory-safe ordering (you hit an OOM last time)
The **retrieval half runs first with no Mistral loaded** (embeddings + vector store only), so
even if the T4 OOMs later, you keep those results. Generation loads Mistral **last**, and there's
a `USE_LOCAL_LLM = False` switch that generates with Claude instead if the GPU won't cooperate.

> **Runtime:** Colab **T4 GPU**. Secrets (key icon): `HF_TOKEN` *and* `ANTHROPIC_API_KEY`.
> Accept the Mistral-7B-Instruct-v0.3 license on Hugging Face. Run top-to-bottom once.

## Setup

In [1]:
!pip install -qU langchain langchain-community langchain-huggingface langchain-qdrant
!pip install -qU qdrant-client sentence-transformers
!pip install -qU transformers accelerate bitsandbytes
!pip install -qU beautifulsoup4 lxml anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.9/109.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 33.3 MB/s eta 0:00:00


In [2]:
import os, re, time, json, textwrap, random
import requests
import numpy as np
from bs4 import BeautifulSoup
from google.colab import userdata

# Two secrets required in Colab (key icon, left panel)
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

import anthropic
from pydantic import BaseModel
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from sentence_transformers import CrossEncoder

SEC_USER_AGENT = "David Schaaf davidschaaf@berkeley.edu"   # edit to your name/email
JUDGE_MODEL = "claude-opus-4-8"
client = anthropic.Anthropic()                            # reads ANTHROPIC_API_KEY
print("Setup ready. Claude judge:", JUDGE_MODEL)

Setup ready. Claude judge: claude-opus-4-8


## Rebuild the Phase-1 pipeline *(provided — you built this in Phase 1)*

Fetch → clean → chunk (each chunk gets a stable `chunk_id` so we can check retrieval exactly) →
embed → index in Qdrant → build a reranked retriever. **No Mistral yet** — none of the retrieval
evaluation below needs a GPU LLM.

In [3]:
# --- fetch + clean + chunk (with chunk_id) ---
COMPANIES = {"AAPL": 320193, "MSFT": 789019, "NVDA": 1045810}

def fetch_latest_10k_html(ticker, cik):
    headers = {"User-Agent": SEC_USER_AGENT}
    recent = requests.get(f"https://data.sec.gov/submissions/CIK{cik:010d}.json",
                          headers=headers).json()["filings"]["recent"]
    i = next(j for j, f in enumerate(recent["form"]) if f == "10-K")
    acc, doc = recent["accessionNumber"][i].replace("-", ""), recent["primaryDocument"][i]
    url = f"https://www.sec.gov/Archives/edgar/data/{cik}/{acc}/{doc}"
    html = requests.get(url, headers=headers).text
    time.sleep(0.5)
    return {"ticker": ticker, "filing_date": recent["filingDate"][i], "html": html}

def clean_filing_text(html):
    soup = BeautifulSoup(html, "lxml")
    for t in soup(["script", "style"]):
        t.decompose()
    return re.sub(r"\s+", " ", soup.get_text(" ")).strip()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
documents = []
for tk, cik in COMPANIES.items():
    print("Fetching", tk, "10-K ...")
    f = fetch_latest_10k_html(tk, cik)
    for chunk in splitter.split_text(clean_filing_text(f["html"])):
        documents.append(Document(page_content=chunk,
            metadata={"chunk_id": len(documents), "ticker": tk, "filing_date": f["filing_date"]}))
print("Total chunks:", len(documents))

Fetching AAPL 10-K ...


/tmp/ipykernel_1402/3087283996.py:16: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(html, "lxml")


Fetching MSFT 10-K ...
Fetching NVDA 10-K ...
Total chunks: 1093


In [ ]:
# --- embed + index + reranked retrieval ---
embeddings = HuggingFaceEmbeddings(model_name="all-mpnet-base-v2")
qdrant = QdrantVectorStore.from_documents(
    documents, embeddings, location=":memory:", collection_name="sec_eval")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_baseline(query, k=10):
    return qdrant.similarity_search(query, k=k)

def retrieve_reranked(query, fetch_k=10, top_k=10):
    cands = qdrant.similarity_search(query, k=fetch_k)
    scores = reranker.predict([(query, d.page_content) for d in cands])
    return [d for _, d in sorted(zip(scores, cands), key=lambda x: x[0], reverse=True)][:top_k]

def format_docs(docs):
    return "\n\n".join(f"[{d.metadata['ticker']}] {d.page_content}" for d in docs)

print("Pipeline ready (no GPU LLM loaded yet).")

---
## 1 · Build a gold evaluation set *(synthetic)*

To measure retrieval you need ground truth: for a known chunk, a question that chunk answers.
We use Claude to generate them — for each sampled chunk, one question answerable **only** from
that chunk. The chunk's `chunk_id` is then the *correct* retrieval for that question. This is a
standard, automatable way to get a labeled eval set without hand-writing Q/A pairs.

In [5]:
# --- PROVIDED: a tiny Claude helper (plain text answer) ---
def ask_claude(prompt, max_tokens=256):
    msg = client.messages.create(
        model=JUDGE_MODEL, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}])
    return msg.content[0].text.strip()

print("Claude helper ready.")

Claude helper ready.


In [13]:
# --- YOU WRITE: generate one question a given chunk uniquely answers ---
# Prompt Claude (via ask_claude) to write ONE question answered ONLY by this excerpt,
# and return just the question text.
def gen_question(chunk_text):
    ### YOUR CODE HERE
    prompt = f"""Context: {chunk_text}

    Given the previous context, write one question that a financial analyst would ask about it.
    The expected answer should be derived ONLY from this excerpt, no outside information.
    Respond only with the question, nothing else. If there is no context, say there is no context.

    Question:"""
    return ask_claude(prompt)

print(f"Context: {documents[len(documents)//2].page_content[:150]}")
print(gen_question(documents[len(documents)//2].page_content))

Context: incurred to support and maintain cloud-based and other online products and services, including datacenter costs and royalties; manufacturing and distr
What types of costs are included in research and development expenses until technological feasibility is reached?


In [14]:
# --- PROVIDED: sample content-rich chunks, build the gold set ---
N_GOLD = 12
random.seed(7)
rich = [d for d in documents if len(d.page_content) > 600
        and sum(c.isalpha() for c in d.page_content) / len(d.page_content) > 0.7]
sampled = random.sample(rich, N_GOLD)

gold = []
for d in sampled:
    gold.append({"question": gen_question(d.page_content),
                 "source_id": d.metadata["chunk_id"], "ticker": d.metadata["ticker"]})
print(gold[0])
print()
print(f"Built {len(gold)} gold Q/A pairs.")


{'question': "What factors will determine the extent to which global pandemics impact the company's business going forward?", 'source_id': 441, 'ticker': 'MSFT'}

Built 12 gold Q/A pairs.


In [15]:
# --- INSTRUMENT: eyeball the gold set ---
for g in gold[:3]:
    src = documents[g["source_id"]].page_content[:120].replace("\n", " ")
    print(f"[{g['ticker']}] Q: {g['question']}")
    print(f"        source chunk {g['source_id']}: {src}...\n")

[MSFT] Q: What factors will determine the extent to which global pandemics impact the company's business going forward?
        source chunk 441: condition, and results of operations. The extent to which global pandemics impact our business going forward will depend...

[AAPL] Q: What percentage of total trade receivables did the Company's third-party cellular network carriers account for as of September 27, 2025?
        source chunk 194: 28, 2024, were as follows (in millions): 2025 2024 Derivative instruments designated as accounting hedges: Foreign excha...

[MSFT] Q: What are the principal foreign currency exposures that the company monitors for foreign exchange risk?
        source chunk 519: matters concerning internal controls and financial reporting. Deloitte & Touche LLP and the internal auditors each have ...



**🎤 Talk-track — Gold set.** *"You can't improve what you don't measure, and RAG's failure
mode is silent — a confident answer over the wrong context. I bootstrap a labeled eval set with an
LLM: generate a question from a known chunk, and now I know exactly which chunk retrieval should
return. It's cheap, repeatable, and gives me ground truth to score against."*

---
## 2 · Retrieval metrics — recall@k & MRR  *(no GPU needed)*

Retrieval is the ceiling on the whole system: if the right chunk never reaches the prompt, no
LLM can answer correctly. Two standard metrics:
- **recall@k** — is the correct chunk in the top-k?
- **MRR** (mean reciprocal rank) — `1/rank` of the correct chunk, averaged. Rewards ranking it
  *higher*, not just *present*.

In [21]:
# --- YOU WRITE: recall@k and reciprocal rank ---
# retrieved_ids: run retrieve_fn(query) and pull each doc's metadata["chunk_id"]
# recall_at_k:   1.0 if source_id is in the top-k ids else 0.0
# reciprocal_rank: 1/(rank) where rank is 1-indexed position of source_id, else 0.0
def retrieved_ids(query, retrieve_fn):
    result = retrieve_fn(query)
    return [d.metadata["chunk_id"] for d in result]

def recall_at_k(source_id, ids, k):
    return float(source_id in ids[:k])

def reciprocal_rank(source_id, ids):
    try:
      return 1 / (ids.index(source_id) + 1)
    except ValueError:
      return 0.0

ids = retrieved_ids(gold[0]["question"], retrieve_baseline)
print("recall@3:", recall_at_k(gold[0]["source_id"], ids, 3),
      "| RR:", round(reciprocal_rank(gold[0]["source_id"], ids), 3))

recall@3: 1.0 | RR: 0.5


In [22]:
# --- INSTRUMENT: score the baseline retriever over the whole gold set ---
rows = [(g["source_id"], retrieved_ids(g["question"], retrieve_baseline)) for g in gold]
for k in (1, 3, 5):
    r = np.mean([recall_at_k(s, ids, k) for s, ids in rows])
    print(f"recall@{k}: {r:.2f}")
print(f"MRR:       {np.mean([reciprocal_rank(s, ids) for s, ids in rows]):.3f}")

recall@1: 0.67
recall@3: 0.83
recall@5: 0.92
MRR:       0.753


**🎤 Talk-track — Retrieval metrics.** *"recall@k tells me whether the answer is even
reachable; MRR tells me whether it's ranked where the LLM will actually use it. This is where I'd
point a dashboard first — retrieval regressions are the cheapest bugs to catch and the most
expensive to miss."*

---
## 3 · Answer evaluation — faithfulness & relevance *(Claude as judge)*

Good retrieval doesn't guarantee a good answer — the LLM can still hallucinate or wander. We score
each generated answer with **Claude as an LLM-judge**:
- **Faithfulness** — is every claim supported by the retrieved context? (hallucination check)
- **Relevance** — does the answer actually address the question?

This is where Mistral loads. **Run the load cell once.** If the T4 OOMs, set `USE_LOCAL_LLM = False`
below to generate with Claude instead (note: that changes *what* you're evaluating).

In [ ]:
# --- PROVIDED: the generator (local Mistral, or Claude fallback) ---
USE_LOCAL_LLM = True   # set False if the T4 OOMs

RAG_TEMPLATE = (" [INST] Answer using ONLY the context. If it's not there, say you don't know.\n\n"
                "Context:\n{context}\n\nQuestion: {question} [/INST] Assistant: ")

if USE_LOCAL_LLM:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
    _mid = "mistralai/Mistral-7B-Instruct-v0.3"
    _qc = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
                             bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    print("Loading Mistral (run once) ...")
    _tok = AutoTokenizer.from_pretrained(_mid)
    _model = AutoModelForCausalLM.from_pretrained(_mid, quantization_config=_qc, device_map="auto")
    _pipe = pipeline("text-generation", model=_model, tokenizer=_tok, max_new_tokens=384,
                     temperature=0.3, do_sample=True, repetition_penalty=1.2, return_full_text=False)
    _pipe.model.config.pad_token_id = _pipe.model.config.eos_token_id
    def _gen(prompt): return _pipe(prompt)[0]["generated_text"].strip()
else:
    def _gen(prompt):
        return client.messages.create(model=JUDGE_MODEL, max_tokens=384,
            messages=[{"role": "user", "content": prompt}]).content[0].text.strip()

def generate_answer(question):
    docs = retrieve_reranked(question, top_k=4)
    context = format_docs(docs)
    return _gen(RAG_TEMPLATE.format(context=context, question=question)), context

print("Generator ready. USE_LOCAL_LLM =", USE_LOCAL_LLM)

In [24]:
# --- PROVIDED: the structured shape Claude must return ---
class Judgment(BaseModel):
    faithfulness: int      # 1 (unsupported) .. 5 (fully supported by context)
    answer_relevance: int  # 1 (off-topic)   .. 5 (directly answers)
    rationale: str

print("Judgment schema ready.")

Judgment schema ready.


In [31]:
# --- YOU WRITE: the Claude LLM-judge ---
# Build a prompt asking Claude to score faithfulness (every claim supported by context?) and
# answer_relevance (does it address the question?) 1-5, given question + context + answer.
# Call client.messages.parse(model=JUDGE_MODEL, max_tokens=512, messages=[...],
#     output_format=Judgment) and return resp.parsed_output.
def judge_answer(question, context, answer):
    ### YOUR CODE HERE
    prompt = f"""Context: {context}

    Question: {question}

    Answer: {answer}""" + """\n\nRead the above Context, Question, and Answer.
    Your task is to assess the faithfullness of the asnwer and its relevance to the question.
    Faithfullness means that every claim of the answer is supported by the context.
    Relevance means that the answer is directly addressing the question.

    Respond with a JSON dictionary:
    {"rationale": "One sentence about your rationale for the faithfullness and answer_relevance score",
     "faithfullness": "integer [1-5]",
     "answer_relevance": "integer [1-5]"
     }"""
    resp = client.messages.parse(model=JUDGE_MODEL, max_tokens=512,
                                  messages=[{"role": "user", "content": prompt}],
                                  output_format=Judgment)

    return resp.parsed_output


_a, _ctx = generate_answer(gold[0]["question"])
print(judge_answer(gold[0]["question"], _ctx, _a))

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


faithfulness=0 answer_relevance=0 rationale='The context explicitly lists only three factors determining pandemic impact (duration/scope, responses, and impact on economic activity including recession/market instability), but the answer includes many unrelated factors (climate, geopolitical, tariffs, product mix) that are not tied to the pandemic question, making it partly unfaithful and only partly relevant.ar_placeholder}, '


In [32]:
# --- INSTRUMENT: judge the pipeline over a small sample ---
SAMPLE = gold[:8]
results = []
for g in SAMPLE:
    ans, ctx = generate_answer(g["question"])
    j = judge_answer(g["question"], ctx, ans)
    results.append({"q": g["question"], "answer": ans, "j": j})

faith = np.mean([r["j"].faithfulness for r in results])
rel   = np.mean([r["j"].answer_relevance for r in results])
print(f"mean faithfulness: {faith:.2f} / 5")
print(f"mean relevance:    {rel:.2f} / 5\n")
ex = results[0]
print("Example —", ex["q"])
print("Answer:", ex["answer"][:300])
print("Judge:", ex["j"].faithfulness, "/", ex["j"].answer_relevance, "—", ex["j"].rationale)

[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=384) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

mean faithfulness: 4.38 / 5
mean relevance:    4.50 / 5

Example — What factors will determine the extent to which global pandemics impact the company's business going forward?
Answer: 1. Duration and scope of the pandemic
2. Governmental, business, and individual responses to the pandemic
3. Impact on economic activity (possibility of recession or financial market instability)
4. Containment measures for the pandemic may amplify other risks mentioned in the text
5. Long-term effe
Judge: 4 / 4 — Points 1-4 are faithful and relevant to the question about pandemic impact factors, but point 5 about climate change is unrelated to the pandemic question and constitutes an irrelevant addition that slightly reduces both scores.


**🎤 Talk-track — LLM-judge.** *"Faithfulness is my hallucination metric: does the answer
stay inside the retrieved evidence? I use a stronger model as the judge and force structured output
so the scores are machine-readable, not prose I have to parse. In a regulated shop this is the
artifact that lets me say 'here's the measured grounding rate,' not 'it looked fine.'"*

---
## 4 · One controlled experiment — does reranking help?

Change **one knob** — reranking on vs. off — hold everything else fixed, and *measure* the effect
on the same gold set. Retrieval-only, so no Mistral and no OOM risk. This is the difference between
*reranking felt better* and *reranking lifted MRR by X, 95% CI [a, b]*.

In [36]:
# --- YOU WRITE: evaluate a retrieval function over the gold set ---
# For each gold item, get retrieved_ids(question, retrieve_fn), then accumulate
# recall_at_k (k=1,3,5) and reciprocal_rank. Return a dict of the means (and keep the
# per-item reciprocal ranks under "_rr" for the CI below).
def evaluate_retrieval(retrieve_fn):
    ### YOUR CODE HERE
    rows = [(g["source_id"], retrieved_ids(g["question"], retrieve_fn)) for g in gold]
    rec1 = [recall_at_k(s, ids, 1) for s, ids in rows]
    rec3 = [recall_at_k(s, ids, 3) for s, ids in rows]
    rec5 = [recall_at_k(s, ids, 5) for s, ids in rows]
    rrs = [reciprocal_rank(s, ids) for s, ids in rows]

    return {
        "recall@1": np.mean(rec1),
        "recall@3": np.mean(rec3),
        "recall@5": np.mean(rec5),
        "MRR": np.mean(rrs),
        "_rr": rrs
    }

base = evaluate_retrieval(retrieve_baseline)
rerank = evaluate_retrieval(lambda q: retrieve_reranked(q, top_k=10))

In [37]:
# --- INSTRUMENT: comparison table + a bootstrap CI on the MRR lift ---
print(f"{'metric':<10}{'baseline':>10}{'rerank':>10}{'delta':>10}")
for m in ("recall@1", "recall@3", "recall@5", "MRR"):
    print(f"{m:<10}{base[m]:>10.3f}{rerank[m]:>10.3f}{rerank[m]-base[m]:>+10.3f}")

# paired bootstrap 95% CI on the per-question MRR difference (the rigor nod)
diffs = np.array(rerank["_rr"]) - np.array(base["_rr"])
rng = np.random.default_rng(0)
boot = [rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(2000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"\nMRR lift from reranking: {diffs.mean():+.3f}  (95% CI [{lo:+.3f}, {hi:+.3f}])")
print("CI excludes 0 → the lift is unlikely to be noise." if lo > 0 or hi < 0
      else "CI includes 0 → with this small gold set, the lift isn't distinguishable from noise.")

metric      baseline    rerank     delta
recall@1       0.667     0.667    +0.000
recall@3       0.833     0.917    +0.083
recall@5       0.917     0.917    +0.000
MRR            0.753     0.792    +0.039

MRR lift from reranking: +0.039  (95% CI [-0.153, +0.242])
CI includes 0 → with this small gold set, the lift isn't distinguishable from noise.


**🎤 Talk-track — The experiment.** *"This is the loop closing. I isolate one change,
hold everything else fixed, and report an effect size with a confidence interval — not a vibe. The
bootstrap CI is a small version of the causal-inference discipline I bring: I care whether a lift
is real or noise before I ship it. Scale the gold set up and this is a real offline A/B."*

---
## Recap — the feedback loop, closed

| Stage | What you measured | Why it matters in the room |
|-------|-------------------|-----------------------------|
| **Gold set** | synthetic Q → known source chunk | ground truth without hand-labeling |
| **Retrieval** | recall@k, MRR | the ceiling on the whole system; cheapest bugs to catch |
| **Answer** | faithfulness, relevance (Claude judge) | hallucination control; a *measured* grounding rate |
| **Experiment** | rerank on/off, effect size + CI | measure, don't eyeball; real vs. noise |

You now have the full diagram from the prep doc — ingestion → embedding → vector store → retrieval
→ LLM → **evaluation** — with the last stage instrumented. That last stage is the differentiator.

**What production adds next** (the monitoring half): log every query's retrieval scores and judge
scores online; watch for **drift** (embedding distribution shift, faithfulness dropping over time);
alert on regressions; add guardrails (PII filters, refuse-on-no-context). Same metrics, running
continuously instead of once.